# Reviewer-Response Agent — Live Demo

This notebook walks through every stage of the pipeline against the sample manuscript + reviews in `data/`. Use it to:

- Verify the install works (`uv sync` first).
- Inspect intermediate outputs (chunks, embeddings, parsed comments, per-comment trace).
- Render the final rebuttal letter inline.

Make sure `.env` exists with `GEMINI_API_KEY` set (or `LLM_PROVIDER=ollama` if you've pointed it at your home server).

## 1. Setup — config + provider

Loads `.env`, picks the LLM backend, and verifies the API key is reachable.

In [1]:
from pathlib import Path
from IPython.display import Markdown, display

from response_agent.config import load_config
from response_agent.llm import get_provider

cfg = load_config()
print(f"Provider:       {cfg.provider}")
if cfg.provider == "gemini":
    print(f"Chat model:     {cfg.gemini_chat_model}")
    print(f"Embed model:    {cfg.gemini_embed_model}")
    print(f"Min interval:   {cfg.gemini_min_interval}s   max retries: {cfg.gemini_max_retries}")
else:
    print(f"Ollama host:    {cfg.ollama_host}")
    print(f"Chat model:     {cfg.ollama_chat_model}")
    print(f"Embed model:    {cfg.ollama_embed_model}")

llm = get_provider(cfg)
print("LLM client ready ✓")

Provider:       gemini
Chat model:     gemini-3.1-flash-lite
Embed model:    gemini-embedding-001
Min interval:   6.5s   max retries: 5
LLM client ready ✓


## 2. Load inputs

In [2]:
from response_agent.ingest import load_manuscript, load_reviews

DATA = Path("data")
manuscript = load_manuscript(DATA / "manuscript.pdf")
reviews    = load_reviews(DATA / "reviews.txt")

print(f"Manuscript: {len(manuscript):,} chars")
print(f"Reviews:    {len(reviews):,} chars")
print("\n--- first 300 chars of manuscript ---")
print(manuscript[:300], "…")

Manuscript: 21,969 chars
Reviews:    1,282 chars

--- first 300 chars of manuscript ---
MERLIN: Multiple Enhanced Representations with LLM
Generated INdices
Anirudh Ravichandran∗
aniravic@amazon.com
Amazon
Seattle, WA, USA
Yidong Zou∗
yid@amazon.com
Amazon
Seattle, WA, USA
Jayapragash Baskar
jbaskar@amazon.com
Amazon
Seattle, WA, USA
Anurag Beniwal
beanurag@amazon.com
Amazon
California …


## 3. Chunk + embed the manuscript

Builds the in-memory ChunkIndex used by the retrieval layer.

In [3]:
from response_agent.chunking import chunk_text
from response_agent.retrieval import build_index

chunks = chunk_text(manuscript)
print(f"{len(chunks)} chunks (chunk_chars=1800, overlap=200)")

index = build_index(chunks, llm)
print(f"Index: {index.embeddings.shape[0]} vectors × {index.embeddings.shape[1]} dims")

14 chunks (chunk_chars=1800, overlap=200)
Index: 14 vectors × 3072 dims


## 4. PARSE — split raw reviews into atomic comments

In [4]:
from response_agent.agents import parse_reviews

comments = parse_reviews(llm, reviews)
print(f"Parsed {len(comments)} atomic comments.\n")
for i, c in enumerate(comments, 1):
    snippet = c['comment'][:120] + ('…' if len(c['comment']) > 120 else '')
    print(f"  [{i:>2}] {c['reviewer']}: {snippet}")

Parsed 1 atomic comments.

  [ 1] Reviewer 1: The evaluation is primarily focused on the Amazon customer service domain; testing the approach on a broader range of da…


## 5. Step through ONE comment end-to-end

Verbose walkthrough of the agent loop on the first comment, so you can see what the Draft → Critique → Verify → Refine path looks like in isolation.

In [5]:
from response_agent.agents import critique, draft_response, adversarial_followup
from response_agent.retrieval import retrieve
from response_agent.verifier import verify_citations

demo_comment = comments[0]
print("REVIEWER:", demo_comment['reviewer'])
print("COMMENT:", demo_comment['comment'])
print()

ctx_chunks = retrieve(index, demo_comment['comment'], llm, k=4)
print(f"Retrieved {len(ctx_chunks)} chunks. First chunk preview:\n")
print(ctx_chunks[0][:300], "…\n")

context = "\n\n---\n\n".join(ctx_chunks)
draft = draft_response(llm, demo_comment['comment'], context)
print("--- DRAFT ---\n", draft, "\n")

verdict = critique(llm, demo_comment['comment'], draft, context)
cite_issues = verify_citations(draft, manuscript)
print("--- CRITIC ---")
print("ok:    ", verdict.get('ok'))
print("issues:", verdict.get('issues') or '—')
print("verifier:", cite_issues or '(no citation issues)')
print()

if not verdict.get('ok') or cite_issues:
    merged = " ".join(filter(None, [verdict.get('issues', ''), cite_issues]))
    refined = draft_response(llm, demo_comment['comment'], context, refine_note=merged)
    print("--- REFINED ---\n", refined)
else:
    print("(no refinement needed)")

REVIEWER: Reviewer 1
COMMENT: The evaluation is primarily focused on the Amazon customer service domain; testing the approach on a broader range of datasets and retrieval tasks would strengthen the generalizability of the findings.

Retrieved 4 chunks. First chunk preview:

cking the real customer input
(and the corresponding ground truth document) that were manually
labeled. In order to gather ground truth relevance information, we
presented the annotators with the query and the top-6 options as
picked from an LLM from an initial candidate pool of 15 candidates
that w …

--- DRAFT ---
 Thank you for your constructive feedback regarding the scope of our evaluation. We agree that testing our multi-index enhancement approach on broader datasets would further demonstrate the generalizability of our findings.

In our current work, we focused on the Amazon customer service domain to address specific industrial requirements, such as the need for high-precision retrieval within strict latency

## 6. Run the full pipeline

Same as `uv run python main.py`. Writes `outputs/rebuttal.md` and `outputs/rebuttal.pdf`.

In [6]:
from response_agent.pipeline import run

out = run(
    manuscript_path=DATA / "manuscript.pdf",
    reviews_path=DATA / "reviews.txt",
    output_path=Path("outputs/rebuttal.md"),
    top_k=4,
    max_refine_passes=1,
    adversarial=False,   # flip to True for the diagnostic follow-ups (doubles LLM calls)
)
print("\nWritten:", out, "and", out.with_suffix('.pdf'))

[1/5] Loading manuscript: data/manuscript.pdf
      manuscript: 21,969 chars   reviews: 1,282 chars
[2/5] Chunking + embedding manuscript
      14 chunks indexed (dim=3072)
[3/5] Parsing reviews into atomic comments
      1 comments extracted
[4/5] Drafting + critiquing each comment
      [ 1/1] Reviewer 1 — refined x1
[5/5] Writing letter to outputs/rebuttal.md
      PDF: outputs/rebuttal.pdf

Written: outputs/rebuttal.md and outputs/rebuttal.pdf


## 7. Render the final rebuttal letter inline

In [7]:
letter = Path("outputs/rebuttal.md").read_text()
print(f"{len(letter):,} chars\n")
display(Markdown(letter))

1,512 chars



# Response to Reviewers

We thank the reviewers for their thoughtful and constructive feedback. Below we address each comment in turn.

---

## Reviewer 1

### Comment 1

> The evaluation is primarily focused on the Amazon customer service domain; testing the approach on a broader range of datasets and retrieval tasks would strengthen the generalizability of the findings.

**Response.** We appreciate the reviewer’s suggestion regarding the evaluation of our approach on a broader range of datasets to demonstrate generalizability. 

We acknowledge that our current study focuses specifically on the Amazon customer service domain to provide a concrete, industrial-scale implementation of a multi-index retrieval system. While we agree that testing on public benchmarks would be a valuable addition, our work is specifically designed to address the unique constraints of customer service retrieval, such as the need for near-real-time index updates and strict latency requirements for production environments. As detailed in Section 2.1, our methodology relies on domain-specific fine-tuning and human-annotated ground truth data tailored to real customer queries, which are essential for maintaining high performance in this specific application. 

We have clarified the scope of our evaluation in Section 2.1, noting that our focus remains on demonstrating the effectiveness of the multi-index enhancement within an industrial setting where latency and domain-specific accuracy are the primary constraints.


## 8. (Optional) Adversarial follow-ups

Generates a hostile-reviewer follow-up per comment. Diagnostic only — these are *not* in the rebuttal letter. Useful before submitting to anticipate what round 2 will look like.

In [8]:
# Re-run with the adversarial agent enabled, but only on the first 3 comments to keep it fast.
for i, c in enumerate(comments[:3], 1):
    ctx = "\n\n---\n\n".join(retrieve(index, c['comment'], llm, k=4))
    draft = draft_response(llm, c['comment'], ctx)
    followup = adversarial_followup(llm, c['comment'], draft, ctx)
    print(f"\n[{i}] {c['reviewer']}")
    print("  comment :", c['comment'][:120], "…")
    print("  follow-up:", followup or '(none — response is airtight)')


[1] Reviewer 1
  comment : The evaluation is primarily focused on the Amazon customer service domain; testing the approach on a broader range of da …
  follow-up: The authors claim that the consistency of results across different bi-encoder models and prompting strategies sufficiently compensates for the lack of dataset diversity. However, consistency within a single, narrow domain does not address the fundamental concern regarding the generalizability of the multi-index enhancement framework to different retrieval tasks or corpus characteristics. Please provide at least one additional evaluation on a standard, non-proprietary IR benchmark (e.g., BEIR) to demonstrate that these performance gains are not merely an artifact of the Amazon customer service dataset.


## Done

- Modular pipeline drove the same prompts as `main.py` and `app.py`.
- Outputs: `outputs/rebuttal.md` + `outputs/rebuttal.pdf`.
- To swap to a local Ollama server: set `LLM_PROVIDER=ollama` in `.env` and restart this kernel — every cell above will then call your Ollama instead, with no code change.